In [5]:
import pandas as pd
import numpy as np
import re

In [6]:
# MovieLense https://movielens.org/ – verze do roku 2000

# načtení dat – filmy
movie_col_names = ['movie_id', 'title', 'genre']
movies = pd.read_csv('./movies_database/movies.dat', sep="::", header=None, names=movie_col_names, engine="python", encoding="latin-1")
movies.head()

,movie_id,title,genre
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
# načtení dat – uživatelé
users_col_names = ['user_id', 'gender', 'age', 'occupation', 'zip']
users = pd.read_csv('./movies_database/users.dat', sep="::", header=None, names=users_col_names, engine="python", encoding="latin-1")
users.head()

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [8]:
# načtení dat – hodnocení
ratings_col_names = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('./movies_database/ratings.dat', sep="::", header=None, names=ratings_col_names, engine="python", encoding="latin-1")
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [9]:
# chceme 100 nejvíce hodnocených filmů od každého žánru. 
# film: název, rok, den, měsíc, žánry a průměrné hodnocení filmu.

In [10]:
# spojíme data do jednoho dataframe
data = pd.merge(movies, ratings)
data.head()

,movie_id,title,genre,user_id,rating,timestamp
0,1,Toy Story (1995),Animation|Children's|Comedy,1,5,978824268
1,1,Toy Story (1995),Animation|Children's|Comedy,6,4,978237008
2,1,Toy Story (1995),Animation|Children's|Comedy,8,4,978233496
3,1,Toy Story (1995),Animation|Children's|Comedy,9,5,978225952
4,1,Toy Story (1995),Animation|Children's|Comedy,10,5,978226474


In [11]:
# to co nepotřebujeme odstraníme
to_drop = ['user_id']
data.drop(columns=to_drop, inplace=True)
data.head()

,movie_id,title,genre,rating,timestamp
0,1,Toy Story (1995),Animation|Children's|Comedy,5,978824268
1,1,Toy Story (1995),Animation|Children's|Comedy,4,978237008
2,1,Toy Story (1995),Animation|Children's|Comedy,4,978233496
3,1,Toy Story (1995),Animation|Children's|Comedy,5,978225952
4,1,Toy Story (1995),Animation|Children's|Comedy,5,978226474


In [12]:
# název filmu a rok rozdělíme na samostatné sloupce
years = []
title = [] 
for values in data['title']:
    years.append(int(re.search(r'(\([1-3][0-9]{3})\)', values).group()[1:5]))
    title.append(re.sub(r'(\([1-3][0-9]{3})\)', '', values))

data['year'] = years
data['title'] = title
data.head()

,movie_id,title,genre,rating,timestamp,year
0,1,Toy Story,Animation|Children's|Comedy,5,978824268,1995
1,1,Toy Story,Animation|Children's|Comedy,4,978237008,1995
2,1,Toy Story,Animation|Children's|Comedy,4,978233496,1995
3,1,Toy Story,Animation|Children's|Comedy,5,978225952,1995
4,1,Toy Story,Animation|Children's|Comedy,5,978226474,1995


In [13]:
# odstraníme year
to_drop = ['year']
data.drop(columns=to_drop, inplace=True)
data.head()

,movie_id,title,genre,rating,timestamp
0,1,Toy Story,Animation|Children's|Comedy,5,978824268
1,1,Toy Story,Animation|Children's|Comedy,4,978237008
2,1,Toy Story,Animation|Children's|Comedy,4,978233496
3,1,Toy Story,Animation|Children's|Comedy,5,978225952
4,1,Toy Story,Animation|Children's|Comedy,5,978226474


In [14]:
# vybereme pro každý film nejnovější záznam
latest_ratings = data.loc[data.groupby('title')['timestamp'].idxmax()]

latest_ratings.head()

,movie_id,title,genre,rating,timestamp
557677,2031,"$1,000,000 Duck",Children's|Comedy,2,1038194385
836127,3112,'Night Mother,Drama,3,1043784591
194734,779,'Til There Was You,Drama|Romance,2,1034029843
566214,2072,"'burbs, The",Comedy,3,1046186717
897087,3420,...And Justice for All,Drama|Thriller,4,1042232468


In [15]:
# spočítáme počet hodnocení a průměrné hodnocení pro každý film
movie_stats = data.groupby('title').agg(num_ratings=('rating', 'count'), mean_rating=('rating', 'mean'))

movie_stats.head()

,num_ratings,mean_rating
title,,
"$1,000,000 Duck",37,3.027027
'Night Mother,70,3.371429
'Til There Was You,52,2.692308
"'burbs, The",303,2.910891
...And Justice for All,199,3.713568


In [16]:
# spojíme počet hodnocení s latest_ratings
data = latest_ratings.merge(movie_stats, on='title')

data.head()

,movie_id,title,genre,rating,timestamp,num_ratings,mean_rating
0,2031,"$1,000,000 Duck",Children's|Comedy,2,1038194385,37,3.027027
1,3112,'Night Mother,Drama,3,1043784591,70,3.371429
2,779,'Til There Was You,Drama|Romance,2,1034029843,52,2.692308
3,2072,"'burbs, The",Comedy,3,1046186717,303,2.910891
4,3420,...And Justice for All,Drama|Thriller,4,1042232468,199,3.713568


In [17]:
# timestamp převedeme na den, měsíc, rok

data['datetime'] = pd.to_datetime(data['timestamp'], unit='s')
data['year'] = data['datetime'].dt.year
data['month'] = data['datetime'].dt.month
data['day'] = data['datetime'].dt.day

data.head()

,movie_id,title,genre,rating,timestamp,num_ratings,mean_rating,datetime,year,month,day
0,2031,"$1,000,000 Duck",Children's|Comedy,2,1038194385,37,3.027027,2002-11-25 03:19:45,2002,11,25
1,3112,'Night Mother,Drama,3,1043784591,70,3.371429,2003-01-28 20:09:51,2003,1,28
2,779,'Til There Was You,Drama|Romance,2,1034029843,52,2.692308,2002-10-07 22:30:43,2002,10,7
3,2072,"'burbs, The",Comedy,3,1046186717,303,2.910891,2003-02-25 15:25:17,2003,2,25
4,3420,...And Justice for All,Drama|Thriller,4,1042232468,199,3.713568,2003-01-10 21:01:08,2003,1,10


In [18]:
# odstraníme timestamp a datetime

to_drop = ['timestamp', 'datetime', 'rating']
data.drop(columns=to_drop, inplace=True)
data.head()

,movie_id,title,genre,num_ratings,mean_rating,year,month,day
0,2031,"$1,000,000 Duck",Children's|Comedy,37,3.027027,2002,11,25
1,3112,'Night Mother,Drama,70,3.371429,2003,1,28
2,779,'Til There Was You,Drama|Romance,52,2.692308,2002,10,7
3,2072,"'burbs, The",Comedy,303,2.910891,2003,2,25
4,3420,...And Justice for All,Drama|Thriller,199,3.713568,2003,1,10


In [19]:
genres = data['genre'].str.split('|')

g = set([])
for i in genres:
    g = g.union(set(i))
    
g

{'Action',
 'Adventure',
 'Animation',
 "Children's",
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western'}

In [20]:
# rozšíříme data
movies = data

for j in g:
    movies[j] = movies['genre'].str.contains(str(j))

In [21]:
movies.drop(columns=['genre'], inplace=True) 
movies.head()

,movie_id,title,num_ratings,mean_rating,year,month,day,Drama,Western,Adventure,...,War,Sci-Fi,Thriller,Romance,Horror,Film-Noir,Crime,Action,Musical,Mystery
0,2031,"$1,000,000 Duck",37,3.027027,2002,11,25,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,3112,'Night Mother,70,3.371429,2003,1,28,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,779,'Til There Was You,52,2.692308,2002,10,7,True,False,False,...,False,False,False,True,False,False,False,False,False,False
3,2072,"'burbs, The",303,2.910891,2003,2,25,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,3420,...And Justice for All,199,3.713568,2003,1,10,True,False,False,...,False,False,True,False,False,False,False,False,False,False


In [22]:
# seznam sloupců, které jsou žánry
genre_columns = list(g)
genre_columns

['Drama',
 'Western',
 'Adventure',
 'Comedy',
 'Documentary',
 "Children's",
 'Animation',
 'Fantasy',
 'War',
 'Sci-Fi',
 'Thriller',
 'Romance',
 'Horror',
 'Film-Noir',
 'Crime',
 'Action',
 'Musical',
 'Mystery']

In [23]:
# rozpustíme žánry do dvou sloupců: genre a is_genre
movies_exploded = movies.melt(id_vars=['movie_id','title','num_ratings','mean_rating','year','month','day'], value_vars=genre_columns,
    var_name='genre', value_name='is_genre')

# vybereme jen ty řádky, kde je žánr True
movies_exploded = movies_exploded[movies_exploded['is_genre']]

# odstraníme is_genre
to_drop = ['is_genre']
movies_exploded.drop(columns=to_drop, inplace=True)

In [24]:
# seřadíme podle počtu hodnocení
data = movies_exploded.sort_values(by='num_ratings', ascending=False)

data.head()

,movie_id,title,num_ratings,mean_rating,year,month,day,genre
126,2858,American Beauty,3428,4.317386,2003,2,27,Drama
11118,2858,American Beauty,3428,4.317386,2003,2,27,Comedy
58077,260,Star Wars: Episode IV - A New Hope,2991,4.453694,2003,2,22,Action
28765,260,Star Wars: Episode IV - A New Hope,2991,4.453694,2003,2,22,Fantasy
10445,260,Star Wars: Episode IV - A New Hope,2991,4.453694,2003,2,22,Adventure


In [25]:
# vybereme 100 nejvice hodnocenych filmu každého žánru
top_movies_by_genre = pd.DataFrame()

for genre in genre_columns:
    genre_movies = data[data['genre'] == genre].sort_values(by='num_ratings', ascending=False).head(100)
    top_movies_by_genre = pd.concat([top_movies_by_genre, genre_movies], ignore_index=True)

In [26]:
top_movies_by_genre.head(101)

,movie_id,title,num_ratings,mean_rating,year,month,day,genre
0,2858,American Beauty,3428,4.317386,2003,2,27,Drama
1,1196,Star Wars: Episode V - The Empire Strikes Back,2990,4.292977,2003,2,27,Drama
2,2028,Saving Private Ryan,2653,4.337354,2003,2,20,Drama
3,593,"Silence of the Lambs, The",2578,4.351823,2003,2,21,Drama
4,608,Fargo,2513,4.254676,2003,2,6,Drama
...,...,...,...,...,...,...,...,...
96,1953,"French Connection, The",861,4.098722,2003,2,19,Drama
97,1246,Dead Poets Society,855,4.005848,2003,2,7,Drama
98,2908,Boys Don't Cry,850,4.014118,2003,1,28,Drama
99,3543,Diner,839,3.922527,2003,2,18,Drama


In [27]:
# test, že skutečně vypisuje 100 filmu konkrétního žánru
drama_movies = top_movies_by_genre[top_movies_by_genre['genre'] == 'Drama']
drama_movies

,movie_id,title,num_ratings,mean_rating,year,month,day,genre
0,2858,American Beauty,3428,4.317386,2003,2,27,Drama
1,1196,Star Wars: Episode V - The Empire Strikes Back,2990,4.292977,2003,2,27,Drama
2,2028,Saving Private Ryan,2653,4.337354,2003,2,20,Drama
3,593,"Silence of the Lambs, The",2578,4.351823,2003,2,21,Drama
4,608,Fargo,2513,4.254676,2003,2,6,Drama
...,...,...,...,...,...,...,...,...
95,3699,Starman,865,3.461272,2003,2,1,Drama
96,1953,"French Connection, The",861,4.098722,2003,2,19,Drama
97,1246,Dead Poets Society,855,4.005848,2003,2,7,Drama
98,2908,Boys Don't Cry,850,4.014118,2003,1,28,Drama
